In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler

C:\Users\Vishal\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Vishal\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [5]:
books = pd.read_csv("../data/raw/books.csv")
ratings = pd.read_csv("../data/raw/ratings.csv")
tags = pd.read_csv("../data/raw/tags.csv")
book_tags = pd.read_csv("../data/raw/book_tags.csv")
to_read = pd.read_csv("../data/raw/to_read.csv")

print("Books:", books.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

Books: (10000, 23)
Ratings: (1048575, 3)
Tags: (34252, 2)


In [7]:
books = books.drop_duplicates()
ratings = ratings.drop_duplicates()
book_tags = book_tags.drop_duplicates()
to_read = to_read.drop_duplicates()

In [25]:

user_std = ratings.groupby("user_id")["rating"].std()

normal_users = user_std[user_std > 0].index

ratings = ratings[ratings["user_id"].isin(normal_users)]

print("Users after outlier removal:", ratings['user_id'].nunique())

Users after outlier removal: 12757


In [9]:
books['authors'] = books['authors'].fillna("Unknown")
books['original_publication_year'] = books['original_publication_year'].fillna(
    books['original_publication_year'].median()
)

In [11]:
scaler = MinMaxScaler()

ratings['rating_normalized'] = scaler.fit_transform(
    ratings[['rating']]
)

In [13]:
user_counts = ratings['user_id'].value_counts()

active_users = user_counts[user_counts >= 5].index

ratings = ratings[ratings['user_id'].isin(active_users)]

In [15]:
book_counts = ratings['book_id'].value_counts()

popular_books = book_counts[book_counts >= 5].index

ratings = ratings[ratings['book_id'].isin(popular_books)]

In [17]:
user_item_matrix = ratings.pivot_table(
    index="user_id",
    columns="book_id",
    values="rating_normalized"
)

In [29]:
# Merge tag names
book_tags = book_tags.merge(tags, on="tag_id", how="left")

# Check column exists
print(book_tags.columns)

# Create tag text
tag_text = book_tags.groupby("goodreads_book_id")["tag_name"].apply(
    lambda x: " ".join(x.astype(str))
)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(tag_text)

tag_matrix = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=tag_text.index,
    columns=tfidf.get_feature_names_out()
)

print(tag_matrix.shape)

Index(['goodreads_book_id', 'tag_id', 'count', 'tag_name_x', 'tag_name_y',
       'tag_name'],
      dtype='str')
(10000, 16611)


In [31]:

book_popularity = ratings.groupby("book_id").size()

books["popularity_score"] = books["book_id"].map(book_popularity)

books["popularity_score"] = books["popularity_score"].fillna(0)

In [21]:
books['content_features'] = (
    books['authors'].astype(str) + " " +
    books['original_publication_year'].astype(str) + " " +
    books['average_rating'].astype(str)
)

In [23]:
cold_start_users = set(ratings['user_id']) - set(user_item_matrix.index)
cold_start_books = set(books['book_id']) - set(user_item_matrix.columns)

print("Cold Start Users:", len(cold_start_users))
print("Cold Start Books:", len(cold_start_books))

Cold Start Users: 0
Cold Start Books: 2736


In [33]:


# Save cleaned ratings
ratings.to_csv("../data/processed/ratings_clean.csv", index=False)

# Save enhanced books dataset
books.to_csv("../data/processed/books_enhanced.csv", index=False)

# Save user-item matrix (Collaborative Filtering)
user_item_matrix.to_pickle("../data/processed/user_item_matrix.pkl")

# Save TF-IDF tag matrix (Content Based Filtering)
tag_matrix.to_pickle("../data/processed/tag_matrix_tfidf.pkl")

# Save content features
books[['book_id','content_features']].to_pickle(
    "../data/processed/content_features.pkl"
)

# Save cold start lists (optional but useful)
import pickle

with open("../data/processed/cold_start_users.pkl","wb") as f:
    pickle.dump(cold_start_users, f)

with open("../data/processed/cold_start_books.pkl","wb") as f:
    pickle.dump(cold_start_books, f)

print("✅ All processed files saved successfully!")

✅ All processed files saved successfully!
